# 09 — The harness: why any of this can be believed twice

The routing is the subject of the project; the measurement machinery is what
makes the routing results worth reading. This notebook demonstrates the five
properties everything else depends on.

| Property | Why it matters |
|---|---|
| **Determinism** | A run repeated must give the same answer, in a different process, on a different day |
| **Content-addressed identity** | Two runs that differ in anything that matters must not be pooled |
| **Write-once storage** | Hours of CARLA must survive a kernel restart |
| **Parallelism that changes nothing** | Speed must not be bought with a different answer |
| **Latency inside the budget** | The planner must never be the reason the bus is standing still |

Runtime: about three minutes.

In [ ]:
# --- Bootstrap ------------------------------------------------------------
# This notebook imports the project as `src.…`, which requires the repository
# root on the import path. `./jupyter-run.sh` puts it there; opening the file
# any other way does not. This cell finds the root by walking up from the
# working directory, so the notebook runs wherever it is launched from and
# whatever folder it sits in.
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Seeded runs only reproduce when Python's string hashing is pinned: several
# policies break ties with min() over a set, whose iteration order is
# randomised per process otherwise. The simulation refuses to run unpinned;
# this says so now rather than after a long cell.
if os.environ.get("PYTHONHASHSEED") in (None, "random"):
    print("WARNING: PYTHONHASHSEED is not pinned, so seeded runs will be "
          "refused.\n         Launch with ./jupyter-run.sh, or set "
          "PYTHONHASHSEED=0.")

print(f"repository root: {ROOT}")

In [ ]:
import os

# How much work this notebook does. Every notebook in this folder has this
# switch and honours the AUTOBUS_SCALE environment variable, so nothing here
# needs editing to run it bigger or smaller.
#
#   "quick"  the default: minutes, same conclusions, wider error bars
#   "full"   the sample sizes behind the figures in the submitted documents
#   "smoke"  a tiny end-to-end check that the notebook runs at all
#
# The outputs saved in this file were produced at "full".
SCALE = os.environ.get("AUTOBUS_SCALE", "quick")
print(f"running at SCALE = {SCALE!r}")

In [ ]:
import matplotlib.pyplot as plt
from cycler import cycler

SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

In [ ]:
# The campus map: the world behind every synthetic result in this project.
# stop_density=0.5 is load-bearing -- the generator's default of 0.15 gives a
# 6-stop map on this seed, on which every policy ties. See notebook 01.
CAMPUS = dict(n_seeds=17, bounds=(0.0, 0.0, 2000.0, 2000.0), seed=42,
              stop_density=0.5)

## 1. The same run twice

Two trials, same seed, same everything. Identical results, to the last
floating-point digit.

In [ ]:
from src.algorithm.ReorderingInsertionPolicy import ReorderingInsertionPolicy
from src.experiment import run_trial
from src.map.graph import PlanarStreetGraph

SETTINGS = dict(seed=7, max_requests=60, demand_interval_s=60.0, capacity=40,
                max_wait_s=900.0, silent=True)


def one_run():
    """Run the standard trial used throughout this notebook."""
    return run_trial(
        policy_factory=lambda g: ReorderingInsertionPolicy(g, max_wait_s=900.0),
        graph=PlanarStreetGraph.generate(**CAMPUS), **SETTINGS)


first, second = one_run(), one_run()
print(f"served       {first.kpi.served_fraction!r}")
print(f"             {second.kpi.served_fraction!r}")
print(f"mean wait    {first.kpi.mean_wait_time!r}")
print(f"             {second.kpi.mean_wait_time!r}")
print(f"\nidentical: {first.wait_times == second.wait_times}")

### And the failure that made this a check rather than an assumption

A seed alone does **not** make a trial reproducible across processes. Several
policies break ties with `min()` over a set of strings, and set iteration order
follows Python's per-process string hash randomisation — so `FixedRoutePolicy`
builds a *literally different cycle* from the same graph depending on how the
process was launched. The baseline comparator drives a different route, and
nothing about it looks wrong.

`run.sh` and `jupyter-run.sh` pin `PYTHONHASHSEED` for exactly this reason, and
the simulation **refuses to run** rather than silently produce an
irreproducible number.

In [ ]:
import os
import subprocess
import sys

ROOT_STR = str(ROOT)
probe = (
    "from src.experiment.trial import require_pinned_hash_seed;"
    "require_pinned_hash_seed('run_trial(seed=7)')"
)
environment = {**os.environ, "PYTHONPATH": ROOT_STR}
environment.pop("PYTHONHASHSEED", None)

completed = subprocess.run(
    [sys.executable, "-c", probe], capture_output=True, text=True,
    env=environment, cwd=ROOT_STR)
print("exit code:", completed.returncode)
print(completed.stderr.strip().splitlines()[-1] if completed.stderr else "")

## 2. What makes two runs "the same run"

A stored run is identified by a hash of **everything that could change its
outcome**: the policy and its settings, the map and its generation parameters,
the demand seed, and every trial parameter — resolved against defaults, so that
passing a parameter explicitly and leaving it at its default give the same
identity rather than two.

The consequence is that the store is a cache with a correctness guarantee: if
an identity is present, that exact simulation has already been run.

In [ ]:
from src.experiment.store import RunIdentity, run_id_for

base = dict(
    policy="Reordering W900",
    # The class is carried separately from the label, so a subclass that
    # forgot to extend params() still cannot share its parent's identity.
    policy_class="ReorderingInsertionPolicy",
    policy_params={"max_wait_s": 900.0},
    world={"map": "planar", "map_seed": 42, "backend": "MockBackend"},
    seed=7,
    trial={"max_requests": 200, "capacity": 40, "max_wait_s": 900.0},
)

changed = {**base, "trial": {**base["trial"], "capacity": 20}}
reordered = {**base, "trial": {"capacity": 40, "max_wait_s": 900.0,
                               "max_requests": 200}}

print(f"identity              {run_id_for(base)[:16]}")
print(f"same, keys reordered  {run_id_for(reordered)[:16]}   <- same run")
print(f"capacity 40 -> 20     {run_id_for(changed)[:16]}   <- different run")

### The lesson that cost the most

The identity must include things that are **not obviously parameters of the
simulation**. The CARLA campaign records how the bus was *steered* and what the
route planner optimised, because when the steering changed, every previously
recorded run kept the same identity — so the campaign skipped the new trials as
"already recorded" and the analysis silently pooled runs from two different
buses.

Nothing errors when this happens. The numbers just quietly stop meaning what
they say. The rule that came out of it:

> **If a change alters what a run would produce, it belongs in the identity —
> even if it is not an argument to the simulation.**

And the corollary the store cannot enforce for you: **changing a policy's code
does not invalidate its recorded runs.** That is a discipline.

## 3. Storage that survives everything

Runs are written one JSON document per run, once, and never modified. A kernel
restart, a crashed campaign or a machine reboot costs at most the run that was
in flight.

In [ ]:
import json
from pathlib import Path
import tempfile

from src.experiment.store import (
    RunRecord, is_recorded, load_runs, save_run)

with tempfile.TemporaryDirectory() as scratch:
    directory = Path(scratch)
    identity = RunIdentity(**base)
    record = RunRecord(
        identity=identity, kpi=first.kpi, tick_count=first.tick_count,
        wait_times=first.wait_times, ride_times=first.ride_times,
        provenance={"note": "written by notebook 09"})

    print(f"before: recorded = {is_recorded(directory, identity)}")
    print(f"written: {save_run(directory, record)}")
    print(f"after:  recorded = {is_recorded(directory, identity)}")
    # Filing it again is a no-op rather than a duplicate: the file name IS
    # the identity, so the skip is a filesystem check, not a query.
    print(f"written again: {save_run(directory, record)}")

    stored = load_runs(directory)[0]
    print(f"\nfiled as {identity.run_id}.json")
    print(f"rebuilt served fraction: "
          f"{stored.as_trial_result().kpi.served_fraction}")
    print("\nthe document holds:")
    for key in json.loads(
            (directory / f"{identity.run_id}.json").read_text()):
        print(f"  {key}")

Each document carries its own identity, its KPIs and its
provenance — including the git commit the run was produced at, which is what
makes "these numbers are stale" a question that can be answered rather than
argued about.

## 4. Parallelism that cannot change an answer

A trial is a pure function of its identity: same inputs, same output, no shared
state. So trials can run in separate processes, and the results are rebuilt
from the store rather than from completion order.

That makes `n_jobs` a scheduling choice and nothing more — which is a claim
worth demonstrating rather than asserting.

In [ ]:
import time

from tqdm.auto import tqdm

from src.experiment import run_campaign

SEEDS = {"smoke": range(2), "quick": range(8), "full": range(16)}[SCALE]
POLICIES = [("Reordering W900",
             lambda g: ReorderingInsertionPolicy(g, max_wait_s=900.0))]
WORLD = {"map": "planar", "n_seeds": CAMPUS["n_seeds"],
         "bounds": list(CAMPUS["bounds"]), "map_seed": CAMPUS["seed"],
         "stop_density": CAMPUS["stop_density"], "backend": "MockBackend",
         "note": "parallelism demonstration"}

timings = {}
# The campaigns themselves run silently here: a progress bar is cheap, but
# this cell is a timing comparison and both arms must carry the same cost.
for n_jobs in tqdm((1, -1), desc="scheduling", unit="setting"):
    with tempfile.TemporaryDirectory() as scratch:
        started = time.perf_counter()
        outcome = run_campaign(
            policies=POLICIES, seeds=SEEDS, world=WORLD,
            graph=PlanarStreetGraph.generate(**CAMPUS),
            directory=Path(scratch), n_jobs=n_jobs, progress=False,
            max_requests=150, demand_interval_s=60.0, capacity=40,
            max_wait_s=900.0)
        timings[n_jobs] = (time.perf_counter() - started,
                           [round(r.kpi.served_fraction, 12)
                            for r in outcome.records])

serial, parallel = timings[1], timings[-1]
print(f"n_jobs=1   {serial[0]:6.1f} s")
print(f"n_jobs=-1  {parallel[0]:6.1f} s   "
      f"({serial[0] / parallel[0]:.1f}x on {os.cpu_count()} cores)")
print(f"\nidentical results: {serial[1] == parallel[1]}")

**Parallelism is refused where it would be wrong.** A world
that declares itself unsafe to parallelise — CARLA, where one server cannot
host a campaign per core — makes `run_campaign` raise rather than quietly
produce runs that interfere with each other.

## 5. A world with no map at all

`ScriptedBackend` replaces the world with a timetable: every leg takes however
long the script says, regardless of geography. There is no graph, no travel
model and no vehicle.

It exists to demonstrate that **nothing above the backend depends on where the
world comes from** — the same agent, policy, demand generator and metrics run
against it and produce ordinary KPIs. It is also the backend the runner's own
tests drive, and a stand-in for CARLA when no server is available.

In [ ]:
from src.simulation.ScriptedBackend import ScriptedBackend

scripted = run_trial(
    policy_factory=lambda g: ReorderingInsertionPolicy(g, max_wait_s=900.0),
    graph=PlanarStreetGraph.generate(**CAMPUS),
    backend_factory=lambda graph, start: ScriptedBackend(
        position=start, leg_durations=(120.0,)),   # every leg is two minutes
    **SETTINGS)

print(f"served {scripted.kpi.served_fraction:.1%}, "
      f"mean wait {scripted.kpi.mean_wait_time:.0f} s "
      f"-- from a world with no geometry in it at all")

## 6. Is the planner ever why the bus is standing still?

The design review requires (§1.1.2) that *routing calculations must be fast
enough that a bus is never forced to wait for them to finish*.

**Identifying the right budget is most of the answer.** The bus plans from
exactly one place — on arrival at a stop — so it is stationary whenever it
plans, and it is stopped anyway for the ~5 s passengers take to board. The tick
rate is not the budget; the dwell is.

The policy is timed by wrapping it, so the run executes exactly the code every
recorded campaign executed. An instrumented planner 5% slower than the real one
would be measuring itself.

In [ ]:
from src.algorithm.DispatchPolicy import DispatchPolicy

DWELL_BUDGET_S = 5.0


class TimedPolicy(DispatchPolicy):
    """
    Times every plan call, and records how big the problem was.

    Recording the size beside the duration is what makes the result
    generalise: latency can then be read as a function of load rather than as
    a fact about one operating point.
    """

    def __init__(self, inner):
        """
        :param inner: The policy whose plan calls are being timed.
        """
        # Set first: __getattr__ delegates through it, so anything reaching
        # it before assignment would recurse forever.
        self._inner = inner
        self.calls = []

    def __getattr__(self, name):
        """Delegate everything not overridden to the wrapped policy."""
        return getattr(self._inner, name)

    def params(self):
        """Report the wrapped policy's parameters unchanged."""
        return self._inner.params()

    def plan(self, state, n_stops):
        """Time one plan call and record the load it faced."""
        started = time.perf_counter()
        route = self._inner.plan(state, n_stops)
        self.calls.append({
            "seconds": time.perf_counter() - started,
            "pending": len(state.pending_requests),
            "onboard": len(state.onboard),
        })
        return route


timed = []


def timed_factory(graph):
    """Build the policy under test, wrapped in a timer."""
    policy = TimedPolicy(ReorderingInsertionPolicy(graph, max_wait_s=1800.0))
    timed.append(policy)
    return policy


# W1800 exceeds passenger patience on purpose: it is the stress case that
# makes schedules longest, not a recommended operating point.
LATENCY_SEEDS = {"smoke": range(1), "quick": range(4),
                 "full": range(10)}[SCALE]

for seed in tqdm(LATENCY_SEEDS, total=len(LATENCY_SEEDS),
                 desc="timing plan calls", unit="trial"):
    run_trial(policy_factory=timed_factory,
              graph=PlanarStreetGraph.generate(**CAMPUS),
              seed=seed, max_requests=200, demand_interval_s=60.0,
              capacity=40, max_wait_s=900.0, silent=True)

import pandas as pd

calls = pd.DataFrame([c for policy in timed for c in policy.calls])
milliseconds = calls["seconds"] * 1000
print(f"{len(calls)} plan calls")
print(f"  p50   {milliseconds.quantile(0.50):8.1f} ms")
print(f"  p99   {milliseconds.quantile(0.99):8.1f} ms")
print(f"  max   {milliseconds.max():8.1f} ms")
print(f"\nbudget  {DWELL_BUDGET_S * 1000:8.0f} ms (the dwell)")
print(f"calls over budget: {(calls['seconds'] > DWELL_BUDGET_S).sum()}")

In [ ]:
figure, axis = plt.subplots(figsize=(7.6, 4.0))
axis.scatter(calls["pending"], calls["seconds"] * 1000, s=6, alpha=0.15,
             color=SERIES[0], linewidths=0)
binned = calls.assign(bucket=(calls["pending"] // 5) * 5).groupby("bucket")
axis.plot(binned["pending"].mean(),
          binned["seconds"].quantile(0.99) * 1000, lw=2, color=SERIES[1],
          label="p99 per bucket")
axis.axhline(DWELL_BUDGET_S * 1000, ls="--", color=INK_SOFT, lw=1.2)
axis.text(0, DWELL_BUDGET_S * 1000 * 1.05, "  dwell budget", fontsize=8,
          color=INK_SOFT)
axis.set_yscale("log")
axis.set_xlabel("pending requests at the moment of planning")
axis.set_ylabel("plan call (ms, log scale)")
axis.set_title("Does latency degrade gracefully, or cliff?")
axis.grid(alpha=0.3)
axis.legend()
figure.tight_layout()

**§1.1.2 is satisfied**, and the way to state it matters: quote
the **max and the p99, never the median**, and name the load it was measured
at. The median describes the calls that were never going to be a problem; the
requirement is about the single call that could make a bus sit at a stop with
its doors shut.

The curve matters as much as the number. A smooth rise extrapolates to loads
this sweep never visited; a cliff would not.

*(At `"full"` scale across 23,536 plan calls on two machines, zero exceeded the
budget; the worst was 2.0 s on the slower one.)*

## 7. And the ordinary kind of testing

Beneath all of the above there is a conventional test suite — the fast,
boring guarantee that the components behave as described.

```bash
./run.sh -m pytest            # 765 tests
```

It covers the graph conversion, the request lifecycle, each policy's specific
promises (including the ones that turned out to be false and were corrected),
the runner's tick order, and the store's identity rules.

## What the harness is for

Every conclusion in the other notebooks rests on being able to say: *this run,
with these settings, produced this number, and running it again produces the
same number.*

The measurement machinery was the largest single piece of engineering in the
project, and the failures it now prevents were all discovered the hard way —
a comparison that pooled two different buses, a baseline that drove a different
route in a different process, a campaign that lost seven runs to an artefact
nobody noticed until the numbers refused to reproduce.

**None of those failures announced themselves.** That is the argument for the
harness.